# PCB / Wafer Surface Defect Detection

**Supported models:**  `faster_rcnn` · `yolo26` · `sme_yolo` · `vit_det` · `vit_mamba` · `rt_detr`

All model code, training loops, and utilities live in `.py` modules.
This notebook is intentionally minimal — just configure, run, and visualise.

## 1 · Install Dependencies

In [ ]:
!pip install -q ultralytics timm mambavision kagglehub pyyaml scikit-learn

## 2 · Download Dataset (Kaggle)

In [ ]:
from google.colab import userdata
import os

os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')

In [ ]:
import kagglehub

path = kagglehub.dataset_download('yidazhang07/bridge-cracks-image')
print('Dataset path:', path)

## 3 · Upload Project Files

Upload all `.py` files and the `models/` folder to `/content/project/`.
Or clone from your repo:

In [ ]:
# Option A: Clone from git (uncomment & set your repo URL)
# !git clone https://github.com/YOUR_USER/PCB_wafer_defect_detection.git /content/project

# Option B: Upload files manually via Colab sidebar,
# then copy them into /content/project
import os
os.makedirs('/content/project', exist_ok=True)
%cd /content/project

In [ ]:
# Verify files are in place
!ls -la *.py models/

## 4 · Configuration

In [ ]:
import config

# ─── Override defaults for this session ─────────────────────────────
config.DATA_DIR = '/root/.cache/kagglehub/datasets/yidazhang07/bridge-cracks-image/versions/1/DeepPCB/PCBData/'
config.YOLO_DATA_DIR = '/content/pcb_yolo_dataset'
config.NUM_EPOCHS = 20
config.BATCH_SIZE = 4
config.USE_PRETRAINED = True

print(f'Device: {config.DEVICE}')
print(f'Data dir: {config.DATA_DIR}')

## 5 · Choose & Train a Model

Change `MODEL_NAME` to switch architectures. Supported values:

| Name | Type | Training Pipeline |
|------|------|-------------------|
| `faster_rcnn` | PyTorch | Custom loop |
| `vit_det` | PyTorch | Custom loop |
| `vit_mamba` | PyTorch | Custom loop |
| `yolo26` | Ultralytics | `model.train()` |
| `sme_yolo` | Ultralytics | `model.train()` |
| `rt_detr` | Ultralytics | `model.train()` |

In [ ]:
MODEL_NAME = 'faster_rcnn'   # <-- CHANGE THIS
print(f'\n>>> Selected model: {MODEL_NAME}')

In [ ]:
from models import create_model

model = create_model(MODEL_NAME, num_classes=config.NUM_CLASSES,
                     pretrained=config.USE_PRETRAINED)
print(type(model))

### 5a · PyTorch Models  (`faster_rcnn`, `vit_det`, `vit_mamba`)

In [ ]:
if MODEL_NAME in ('faster_rcnn', 'vit_det', 'vit_mamba'):
    from dataset import create_dataloaders
    from training import train_model

    train_loader, val_loader, test_loader = create_dataloaders(
        config.DATA_DIR, batch_size=config.BATCH_SIZE,
    )
    print(f'Train: {len(train_loader.dataset)}  Val: {len(val_loader.dataset)}  Test: {len(test_loader.dataset)}')

    history = train_model(
        model, train_loader, val_loader,
        num_epochs=config.NUM_EPOCHS,
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
        device=config.DEVICE,
        save_path=config.BEST_MODEL_PATH,
    )
else:
    print(f'{MODEL_NAME} uses Ultralytics trainer — skip to next cell.')

### 5b · Ultralytics Models  (`yolo26`, `sme_yolo`, `rt_detr`)

These need YOLO-format data. Run the conversion first.

In [ ]:
if MODEL_NAME in ('yolo26', 'sme_yolo', 'rt_detr'):
    from utils import convert_deeppcb_to_yolo

    convert_deeppcb_to_yolo(config.DATA_DIR, config.YOLO_DATA_DIR)
    data_yaml = f'{config.YOLO_DATA_DIR}/data.yaml'
    print(f'YOLO dataset: {data_yaml}')
else:
    print('Not an Ultralytics model — skip.')

In [ ]:
if MODEL_NAME in ('yolo26', 'sme_yolo', 'rt_detr'):
    results = model.train(
        data=data_yaml,
        epochs=config.NUM_EPOCHS,
        imgsz=config.IMG_SIZE,
        batch=config.BATCH_SIZE,
    )
else:
    print('Not an Ultralytics model — skip.')

## 6 · Visualise Results

In [ ]:
if MODEL_NAME in ('faster_rcnn', 'vit_det', 'vit_mamba'):
    from visualization import plot_training_history
    plot_training_history(history)
else:
    # Ultralytics stores results in runs/ — display the curves
    from IPython.display import Image, display
    import glob
    results_dir = sorted(glob.glob('runs/detect/train*/'))[-1]
    for img_name in ['results.png', 'confusion_matrix.png']:
        img_path = f'{results_dir}{img_name}'
        if os.path.exists(img_path):
            display(Image(filename=img_path))

In [ ]:
if MODEL_NAME in ('faster_rcnn', 'vit_det', 'vit_mamba'):
    import torch
    from visualization import plot_augmented_sample

    model.load_state_dict(torch.load(config.BEST_MODEL_PATH))
    model.to(config.DEVICE).eval()

    data_iter = iter(test_loader)
    images, targets = next(data_iter)

    with torch.no_grad():
        preds = model([img.to(config.DEVICE) for img in images])

    # Show first prediction
    plot_augmented_sample(images[0], preds[0])

## 7 · Download Best Model

In [ ]:
from google.colab import files

if MODEL_NAME in ('faster_rcnn', 'vit_det', 'vit_mamba'):
    files.download(config.BEST_MODEL_PATH)
else:
    import glob
    best_pt = sorted(glob.glob('runs/detect/train*/weights/best.pt'))[-1]
    files.download(best_pt)